# Wayanad Landslide Safety RAG — One-time Indexing

This notebook reads `notebooks/Landslide_safety.pdf`, splits the extracted text into chunks,
creates embeddings with `intfloat/e5-base-v2`, and uploads them to Pinecone.

**Run this notebook when the PDF changes or when you need to rebuild the index.**
The Django `app/observations/rag.py` file only retrieves existing vectors; it does not index the PDF.

Expected project layout:

```text
project-root/
├── .env
├── app/observations/rag.py
└── notebooks/
    ├── Landslide_safety.pdf
    └── landslide_rag_index.ipynb
```

## 1. Install dependencies (if needed)

Run this cell once in the notebook environment.

In [1]:
# %pip install --upgrade "transformers<5"

In [2]:
%pip show sentence-transformers transformers torch torchvision

Name: sentence-transformers
Version: 6.2.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License-Expression: Apache-2.0
Location: C:\Users\91904\AppData\Roaming\Python\Python312\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, tokenizers, torch, tqdm, transformers, typing_extensions
Required-by: 
---
Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\91904\AppData\Roaming\Python\Python312\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safeten

In [3]:
# %pip install -U python-dotenv pypdf sentence-transformers pinecone

## 2. Imports and project paths

In [4]:
import hashlib
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader

# This notebook is expected to be saved/run from the project's notebooks/ folder.
PROJECT_DIR = Path.cwd().resolve().parent
PDF_PATH = PROJECT_DIR / "notebooks" / "Landslide_safety.pdf"
ENV_PATH = PROJECT_DIR / ".env"

EMBEDDING_MODEL = "intfloat/e5-base-v2"
EMBEDDING_DIMENSION = 768
INDEX_NAME = "wayanad-landslide-safety"
NAMESPACE = "landslide-safety-v1"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

load_dotenv(ENV_PATH)
print("Project root:", PROJECT_DIR)
print("PDF exists:", PDF_PATH.is_file())

Project root: C:\Users\91904\Desktop\ksfe\AI\CODE\Django_React_AI_project\wayanad-landslide-early-warning-system
PDF exists: True


## 3. Check the environment variables

In [5]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

if not PINECONE_API_KEY:
    raise ValueError(f"PINECONE_API_KEY was not found in {ENV_PATH}. Add it to .env.")
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Safety PDF not found: {PDF_PATH}")

print("Pinecone API key found.")
print("Cloud:", PINECONE_CLOUD)
print("Region:", PINECONE_REGION)

Pinecone API key found.
Cloud: aws
Region: us-east-1


## 4. Extract text from each PDF page

In [6]:
reader = PdfReader(PDF_PATH)
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").replace("\uf0b7", "-").strip()
    if text:
        pages.append((page_number, text))

if not pages:
    raise ValueError("No selectable text was extracted. If this is a scanned PDF, run OCR first.")

print(f"Extracted text from {len(pages)} pages.")
print("First page preview:")
print(pages[0][1][:700])

Extracted text from 1 pages.
First page preview:
LANDSLIDE 
 
Before & During 
 
- Avoid building houses near steep slopes, closes to mountain edges, near drainage ways or 
along natural erosion valleys. 
- Become familiar with land around you. Avoid areas with 
debris flow.  
- In mudflow areas, build channels to direct the flow around 
buildings. 
- Stay alert and awake. Stay Calm and do not panic . Stay 
together. 
- Listen for unusual sounds  that might 
indicate moving debris, such as trees 
cracking or boulders knocking together. 
- Move away from the path of a landslide or 
debris flow as quickly as possible. 
- Avoid river valleys and low-lying areas.  
- If you are near a stream or channel, be alert 
for any sudden increase or dec


## 5. Split page text into overlapping chunks

In [7]:
def create_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if chunk_size < 1 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

chunks = [
    {"page": page_number, "text": chunk}
    for page_number, page_text in pages
    for chunk in create_chunks(page_text)
]

if not chunks:
    raise ValueError("No chunks were created from the PDF.")

print("Total chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

Total chunks: 1
First chunk:
{'page': 1, 'text': 'LANDSLIDE Before & During - Avoid building houses near steep slopes, closes to mountain edges, near drainage ways or along natural erosion valleys. - Become familiar with land around you. Avoid areas with debris flow. - In mudflow areas, build channels to direct the flow around buildings. - Stay alert and awake. Stay Calm and do not panic . Stay together. - Listen for unusual sounds that might indicate moving debris, such as trees cracking or boulders knocking together. - Move away from the path of a landslide or debris flow as quickly as possible. - Avoid river valleys and low-lying areas. - If you are near a stream or channel, be alert for any sudden increase or decrease in water flow and notice whether the water changes from clear to muddy. It is a sign of mudslide/landslide nearby. - Inform nearest tehsil/ district headquarters. - Plant trees and mesh the hillocks to protect soil from eroding. After - Go to designated public shelter

## 6. Create embeddings

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
passages = [f"passage: {item['text']}" for item in chunks]

embeddings = embedding_model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding count:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))
assert len(embeddings[0]) == EMBEDDING_DIMENSION

C:\Users\91904\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.05s/it]

Embedding count: 1
Embedding dimension: 768


## 7. Create or connect to the Pinecone index

In [9]:
pc = Pinecone(api_key=PINECONE_API_KEY)
existing_indexes = pc.list_indexes().names()

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print("Created index. Waiting for it to become ready...")
    deadline = time.monotonic() + 120
    while time.monotonic() < deadline:
        description = pc.describe_index(INDEX_NAME)
        status = description.get("status", {}) if isinstance(description, dict) else getattr(description, "status", {})
        ready = status.get("ready", False) if isinstance(status, dict) else getattr(status, "ready", False)
        if ready:
            break
        time.sleep(1)
    else:
        raise TimeoutError(f"Pinecone index {INDEX_NAME!r} did not become ready in 120 seconds.")
else:
    print("Using existing index:", INDEX_NAME)

description = pc.describe_index(INDEX_NAME)
dimension = description.get("dimension") if isinstance(description, dict) else getattr(description, "dimension", None)
if dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Index dimension mismatch: expected {EMBEDDING_DIMENSION}, got {dimension}. "
        "Create a new index with the correct dimension or use a matching embedding model."
    )

index = pc.Index(INDEX_NAME)
print("Index ready:", INDEX_NAME)

Using existing index: wayanad-landslide-safety
Index ready: wayanad-landslide-safety


## 8. Upload vectors to the namespace

In [10]:
vectors = []
for number, (item, embedding) in enumerate(zip(chunks, embeddings), start=1):
    vectors.append({
        "id": f"chunk-{number:05d}",
        "values": embedding.tolist(),
        "metadata": {
            "text": item["text"],
            "page": item["page"],
        },
    })

# Upsert in batches. Re-running this notebook overwrites vectors with matching IDs.
BATCH_SIZE = 100
for start in range(0, len(vectors), BATCH_SIZE):
    index.upsert(
        vectors=vectors[start:start + BATCH_SIZE],
        namespace=NAMESPACE,
    )

print(f"Uploaded {len(vectors)} vectors to namespace {NAMESPACE!r}.")

Uploaded 1 vectors to namespace 'landslide-safety-v1'.


## 9. Verify the indexed vectors

In [12]:
stats = index.describe_index_stats()
print(stats)

namespace_stats = stats.get("namespaces", {}).get(NAMESPACE, {}) if isinstance(stats, dict) else {}
vector_count = namespace_stats.get("vector_count", 0) if isinstance(namespace_stats, dict) else getattr(namespace_stats, "vector_count", 0)
print("Vector count in namespace:", vector_count)

# if not vector_count:
#     raise RuntimeError("No vectors were found after upload. Check Pinecone and rerun the upload cell.")

DescribeIndexStatsResponse(dimension=768, total_vector_count=2, metric='cosine', namespaces=2)
Vector count in namespace: 0


## 10. Optional retrieval test

This checks vector retrieval only; the Django app uses Ollama separately to generate an answer.

In [13]:
test_question = "What should people do during a landslide?"
test_embedding = embedding_model.encode(
    f"query: {test_question}",
    normalize_embeddings=True,
    show_progress_bar=False,
)

test_results = index.query(
    vector=test_embedding.tolist(),
    top_k=3,
    namespace=NAMESPACE,
    include_metadata=True,
)

for match in test_results.get("matches", []):
    metadata = match.get("metadata", {}) or {}
    print(f"Score: {match.get('score', 0):.4f} | Page: {metadata.get('page')}")
    print(metadata.get("text", "")[:500])
    print("-" * 80)

Score: 0.8876 | Page: 1
LANDSLIDE Before & During - Avoid building houses near steep slopes, closes to mountain edges, near drainage ways or along natural erosion valleys. - Become familiar with land around you. Avoid areas with debris flow. - In mudflow areas, build channels to direct the flow around buildings. - Stay alert and awake. Stay Calm and do not panic . Stay together. - Listen for unusual sounds that might indicate moving debris, such as trees cracking or boulders knocking together. - Move away from the path
--------------------------------------------------------------------------------


## Notes

- Keep `.env` in the project root. Do not commit it to Git.
- Required variable: `PINECONE_API_KEY`.
- Optional variables: `PINECONE_CLOUD=aws`, `PINECONE_REGION=us-east-1`.
- `rag.py` and this notebook must use the same index name and namespace.
- If you change the PDF substantially, rerun the notebook. If you want a clean rebuild, use a new namespace or carefully delete the old namespace's vectors first.
- This is a learning prototype, not an official early-warning system. Do not use its output for public-safety decisions without expert validation.